<a href="https://colab.research.google.com/github/GokulKrishna2017/LangChain/blob/main/Day_3_Parallel_Chains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Dependencies

In [ ]:
!pip install -q \
langchain \
langchain-community \
transformers \
accelerate \
bitsandbytes

###Imports

In [ ]:
from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

from langchain_community.llms import HuggingFacePipeline

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

###Configuring Quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb__4bit_use_double_quant=True
)

Loading the Model

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model =  AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto"
)



Creating HuggingFace Pipeline

In [ ]:
pipe = pipeline(
    task = "text-generation",
    model = model,
    tokenizer = tokenizer
)

##Converting Pipeline to LangChain LLM

In [ ]:
llm = HuggingFacePipeline(
    pipeline = pipe
)

##Promt
###Notes Prompt

In [ ]:
prompt1 = PromptTemplate(
    template= "Generate short and simple notes from the following text. \n\n {text}",
    input_variables = ["text"]
)

###Quiz Prompt

In [ ]:
prompt2 = PromptTemplate(
    template = "Generate 5 short question- answer pairs from the following text.\n\n{text}",
    input_variables= ["text"]
)

Merging the prompt

In [ ]:
prompt3 = PromptTemplate(
    template="create a clean study guide.\n\n Notes:\n{notes}\n\nQuiz:\n{quiz}",
    input_variables= ["notes","quiz"]
)

##create Output Parser

In [ ]:
parser = StrOutputParser()

Creating parallel chain

In [ ]:
parallel_chain = RunnableParallel(
    {
        "notes": prompt1 | llm | parser,
        "quiz": prompt2 | llm | parser
    }
)

##Creating Merge chain

In [ ]:
merge_chain = prompt3 | llm | parser

Combining Everything

In [ ]:
chain = parallel_chain | merge_chain

##Text

In [ ]:
text = """
Support vector machines (SVMs) are a set of supervised learning methods used
for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function
(called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified.
"""

##Running the chain

In [ ]:
result = chain.invoke(
    {"text": text}
    )

In [ ]:
print(result)